In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="2,3"
import torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig, AutoModelForCausalLM
import faiss
import numpy as np
from tqdm import tqdm
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Tokenizer 및 임베딩 모델 로드 (LLM2Vec)
tokenizer_embed = AutoTokenizer.from_pretrained("McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp")
# padding token이 없어서 eos_token을 padding token으로 설정
tokenizer_embed.pad_token = tokenizer_embed.eos_token
# Quantization 설정 (4-bit)
quantization_config = BitsAndBytesConfig(load_in_4bit=True)
# Quantized 인코더 모델 로드
model_embed = AutoModel.from_pretrained(
    "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp",
    quantization_config=quantization_config,
    device_map="auto"  # 자동으로 적절한 디바이스 할당
)

# 모델을 명시적으로 .to(device)로 옮길 필요 없음, 이미 올바른 디바이스로 할당됨
model_embed.eval()  # 평가 모드로 전환
